<a href="https://colab.research.google.com/github/SemalDeSilva/TEAI/blob/Sooriyaarachchi-N.D-IT22254702-withering_stage_detect/withering_stage_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing import image


In [ ]:
BASE_DIR = "/content/drive/MyDrive/tea_dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print("Train:", os.path.exists(TRAIN_DIR))
print("Val  :", os.path.exists(VAL_DIR))
print("Test :", os.path.exists(TEST_DIR))


Train: True
Val  : True
Test : True


In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16    # change to 8 if GPU memory issue
EPOCHS = 20


In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)


Found 417 files belonging to 6 classes.
Found 89 files belonging to 6 classes.
Found 94 files belonging to 6 classes.


In [ ]:
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("NUM_CLASSES:", NUM_CLASSES)


Classes: ['over_withered', 'raw', 'roll', 'surface_moisture', 'toss', 'well_withered']
NUM_CLASSES: 6


In [ ]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.prefetch(tf.data.AUTOTUNE)


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
], name="data_augmentation")


In [ ]:
base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,313 (16.08 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 149s 5s/step - accuracy: 0.4239 - loss: 1.4391 - val_accuracy: 0.6404 - val_loss: 0.8091
Epoch 2/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.6767 - loss: 0.7273 - val_accuracy: 0.7416 - val_loss: 0.6501
Epoch 3/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.7958 - loss: 0.5913 - val_accuracy: 0.7303 - val_loss: 0.6723
Epoch 4/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - accuracy: 0.7941 - loss: 0.5228 - val_accuracy: 0.7191 - val_loss: 0.6545
Epoch 5/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.7855 - loss: 0.4842 - val_accuracy: 0.7416 - val_loss: 0.6745
Epoch 6/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.8571 - loss: 0.3998 - val_accuracy: 0.7191 - val_loss: 0.6373
Epoch 7/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.8379 - loss: 0.4140 - val_accuracy: 0.7416 - val_loss: 0.6428
Epoch 8/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 85s 2s/step - accuracy: 0.8179 - loss: 0.4196 - val_accuracy: 0.7528 - val_loss

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Accuracy:", test_accuracy)


6/6 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - accuracy: 0.8790 - loss: 0.3123
Test Accuracy: 0.7765957713127136


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/withering_stage_model.keras"
model.save(MODEL_PATH)
print("Model saved at:", MODEL_PATH)


Model saved at: /content/drive/MyDrive/withering_stage_model.keras


In [ ]:
def predict_image(image_path):
    img = image.load_img(image_path, target_size=IMAGE_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    idx = int(np.argmax(preds))

    return class_names[idx], float(preds[idx])


In [ ]:
import os

img_path = "/content/drive/MyDrive/tea_dataset/val/surface_moisture/sample_080_20251222_222659_raw.jpg"

true_class = os.path.basename(os.path.dirname(img_path))
print("TRUE CLASS:", true_class)


TRUE CLASS: surface_moisture
